#DBUtils Configuration (Config)

In [0]:
# Section 1: Widgets & Configuration

import pandas as pd
import os
from pyspark.sql.functions import col

# Setup widgets for dynamic path resolution
dbutils.widgets.text("project_catalog", "vstone_catalog", "1. Target Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema Name")
dbutils.widgets.text("landing_volume", "landing", "3. Landing Volume")
dbutils.widgets.text("chunks_volume", "chunks", "4. Chunks Volume")

# Fetch values into variables
CATALOG = dbutils.widgets.get("project_catalog")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
LANDING_VOL = dbutils.widgets.get("landing_volume")
CHUNKS_VOL = dbutils.widgets.get("chunks_volume")

# Construct dynamic paths using Unity Catalog Volume syntax
CHUNKS_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{CHUNKS_VOL}"
SOURCE_FILE = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{LANDING_VOL}/1_main.csv"

# Integrity & Format Logic

In [0]:
# Section 2: Logic (Integrity, Distribution & Format Validation)

print(f"{'='*80}\n STARTING DATA INTEGRITY & FORMAT VALIDATION \n{'='*80}")

try:
    # --- TEST 1: SERVERLESS DATA INTEGRITY VALIDATION ---
    print("\n[TEST 1] Source vs Chunks Row Count...")
    source_total = spark.read.format("csv").option("header", "true").load(SOURCE_FILE).count()
    print(f"Source Total: {source_total:,}")

    chunk_total = 0
    actual_chunks = [f for f in os.listdir(CHUNKS_PATH) if "1_main_chunk" in f]
    
    for chunk_file in sorted(actual_chunks):
        full_path = os.path.join(CHUNKS_PATH, chunk_file)
        
        if chunk_file.endswith(".csv"):
            c = spark.read.format("csv").option("header", "true").load(full_path).count()
            chunk_total += c
            print(f"    - CSV: {chunk_file:25s} | Rows: {c:,}")
        elif chunk_file.endswith(".json"):
            c = spark.read.option("multiLine", "true").json(full_path).count()
            chunk_total += c
            print(f"    - JSON: {chunk_file:25s} | Rows: {c:,}")
        elif chunk_file.endswith(".xml"):
            c = spark.read.text(full_path).filter(col("value").contains("<record>")).count()
            chunk_total += c
            print(f"    - XML: {chunk_file:25s} | Rows: {c:,}")

    print(f"Chunks Total : {chunk_total:,}")
    if source_total != chunk_total:
        raise Exception(f"Row mismatch! Difference: {abs(source_total - chunk_total):,} rows")
    else:
        print(" SUCCESS: Row counts match!")

    # --- TEST 2: PERCENTAGE DISTRIBUTION (50/20/20/10 Split) ---
    print("\n[TEST 2] Split Distribution Check...")
    chunk1_count = spark.read.option("header", "true").csv(f"{CHUNKS_PATH}/1_main_chunk_1.csv").count()
    actual_pct = (chunk1_count / source_total) * 100
    print(f"Actual Distribution: {actual_pct:.2f}% (Target: 50%)")
    
    assert 48 <= actual_pct <= 52, f"Split logic failed! Chunk 1 is {actual_pct}%"
    print(" Distribution Logic Verified.")

    # --- TEST 3: FORMAT & SCHEMA VALIDATION ---
    print("\n[TEST 3] Format & Downstream Compatibility...")

    # 3A: JSON Parsing Test
    df_json = spark.read.option("multiLine", "true").json(f"{CHUNKS_PATH}/1_main_chunk_3.json")
    assert df_json.count() > 0, "JSON file is empty!"
    print(" Chunk 3 JSON: Valid and readable.")

    # 3B: XML Column Naming Convention
    chunk4_csv_path = f"{CHUNKS_PATH}/1_main_chunk_4.csv"
    if os.path.exists(chunk4_csv_path):
        df_xml_check = pd.read_csv(chunk4_csv_path)
        bad_columns = [c for c in df_xml_check.columns if " " in c or "." in c]
        assert len(bad_columns) == 0, f"XML format risk! Found invalid columns: {bad_columns}"
        print(" XML Convention Verified.")
    else:
        xml_sample = spark.read.text(f"{CHUNKS_PATH}/1_main_chunk_4.xml").limit(10).collect()
        print(" XML Structure Check: Accessibility verified.")

except Exception as e:
    print(f" TEST FAILED: {str(e)}")
    raise e

print(f"\n{'='*80}\n ALL TESTS COMPLETE: PIPELINE IS STABLE \n{'='*80}")